In [5]:
#Creating a simple financial bot using Langchain, Langgraph and Groq

!pip install langchain langgraph langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 7.6 MB/s eta 0:00:00


In [4]:
!pip install langchain_groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.4 MB/s eta 0:00:00


In [ ]:
#importing packages
import os
import re
import json
from typing import List, Dict, Any, Optional, TypedDict
from langgraph.graph import StateGraph
from langchain_groq import ChatGroq
from google.colab import userdata

#Step 3
os.environ['GROQ_API_KEY']=userdata.get('GROQ_API_KEY')

#Step 4
llm=ChatGroq(model_name="llama-3.3-70b-versatile",
             groq_api_key=os.environ['GROQ_API_KEY']
             )
#Step 5 Create an LLM function

def ask_llm(prompt:str)->Any:
  response=llm.invoke(prompt)
  return response.content.strip()


#Step 6 State Type State Definition
#This defines what data flow between nodes(agents) in Langgraph

class BotState(TypedDict):
  user_input: str       ##what user typed
  intent: Optional[str]
  data: Optional[str]
  expenses: List[Dict[str, Any]] ##List of Expenses
  hitl_flag: bool


##Defind risky financial behaviour

def is_high_risk(text:str) -> bool:
  risky_keywords =[
      "all_in","sell my house", "crypto all","bet everything","withdraw my entire savings","theft","cancel"

  ]
  return any (k in text for k in risky_keywords)


##Extract number from user input

def parse_amount(text:str):
  ##find numbers
  match=re.search(r"\\d+",text)
  return float(match.group()) if match else None

##Creating nodes(core logic)

#Creating 6 Agents

#Intent
#Expense
#Budgent Node
#Advice Node
#Risk Node
#Fallback node

#First Agent Detecting intent

def node_intent(state: BotState)-> BotState:
  text=state.get("user_input","")

  #Check if the user input is something risky

  state["hitl_flag"]=is_high_risk(text.lower())

  #send to LLM for classification

  prompt=f"""
  Classify into one word : expense,budget,advice, unknown.

  Examplese:
  "Add 200" -> Expense
  "Total Spend?" -> budget
  "How to Save?" -> advice

  User: "{text}"
  Answer:
  """

  ##Invoke LLM to get prediction on intent

  intent=ask_llm(prompt).lower().strip()

  ##Safety check for unknown

  if intent not in ["expense","budget","advice"]:
    intent="unknown"

  state["intent"]=intent

  return state

## 2nd note expense node

def node_expense(state: BotState)-> BotState:
  amt=parse_amount(state.get("user_input",""))

  #If no amount is found

  if amt is None:
    state["data"]="please enter any amount (e.g. Add 1000)"
    return state

  ##Add expense
  state.setdefault("expenses",[]).append({"amount":amt})


  ##Return to user

  state["data"] =f"added {amt} to your expense"
  return state


##3rd node...budget node

def node_budget(state: BotState) -> BotState:
  exps=state.get("expenses",[])

  ###if no expense -> inform user
  if not exps:
    state["data"]="please add some expenses first"
    return state

  total=sum(exp["amount"]for exp in exps)
  #return to the user
  state["data"] =f"your total expense is {total}"
  return state

### 4th advice(LLM will recommend you)

def node_advice(state: BotState) -> BotState:
  prompt=f"""
  Give some money saving tips.

  User: "{state['user_input']}"

  """

  ###Generate some helpfull tips using LLM

  state["data"]=ask_llm(prompt)
  return state

##5th Node risk Node

def node_hitl(state: BotState) -> BotState:
  state["data"] ="High risk financial transaction detected"
  return state

##6th Fall back node

def node_fallback(state: BotState) -> BotState:
  state["data"]="I'm sorry, I didn't understand that."
  return state



######################END OF NODES LOGIC############################

##Decision Logic
def choose_next(state: BotState)-> str:
  if state.get("hitl_flag"):
    return "hitl"
  return state.get("intent","fallback")


###LangGraph Workflow

builder=StateGraph(BotState)

#Add notdes(steps in workflow)

builder.add_node("intent", node_intent)
builder.add_node("expense", node_expense)
builder.add_node("budget", node_budget)
builder.add_node("advice", node_advice)
builder.add_node("hitl", node_hitl)
builder.add_node("fallback", node_fallback)


###Defind starting point

builder.set_entry_point("intent")

##Define routing logic

builder.add_conditional_edges(
    "intent",
    choose_next,
    {
        "expense":"expense",
        "budget":"budget",
        "advice":"advice",
        "hitl":"hitl",
        "unknown":"fallback",

    },
)


## compile it

graph=builder.compile()


##Run chat loop

def run_chat():
  print("Simple finance bot\n")

  ###initial state(memory)###

  state: BotState={
      "expenses":[],
      "hitl_flag":False
  }

  while True:
    msg=input("You define your financial scenario:")

  ##exit condition
  #Exit and quit to exit the loop

    if msg.lower() in ("exit","quit"):
      print("Bot Bye:")
      break

  ###state state with user input

    state["user_input"] =msg

  ##run langraph
    state=graph.invoke(state)

  ##print response

    print("Bot:", state["data"])
    print()

#### start chatbot
run_chat()


Simple finance bot

You define your financial scenario:sell my house
Bot: High risk financial transaction detected

You define your financial scenario:expense 500
Bot: please enter any amount (e.g. Add 1000)

You define your financial scenario:travel 200
Bot: please enter any amount (e.g. Add 1000)

You define your financial scenario:total expense 700
Bot: please add some expenses first

You define your financial scenario:earning 1000
Bot: please add some expenses first

You define your financial scenario:what is my budget
Bot: please add some expenses first

You define your financial scenario:Give me some financial tips
Bot: Here are some valuable financial tips to help you save money:

1. **Create a budget**: Track your income and expenses to understand where your money is going. Make a budget that accounts for all necessary expenses, savings, and debt repayment.
2. **Prioritize needs over wants**: Distinguish between essential expenses (needs) and discretionary spending (wants). Be 